In [1]:
!pip install --quiet gradio python-dotenv openai

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

for folder in [Path.cwd()] + list(Path.cwd().parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(dotenv_path=env, override=True)
        print(f"📄 Loaded: {env}")
        break

for k in ["OPENROUTER_API_KEY", "GROQ_API_KEY", "DEEPSEEK_API_KEY"]:
    v = os.getenv(k, "")
    print(f"{'✅' if v else '❌'} {k}")

📄 Loaded: c:\Users\HP\OneDrive\Desktop\jekacode-ai-engineering\.env
✅ OPENROUTER_API_KEY
✅ GROQ_API_KEY
✅ DEEPSEEK_API_KEY


In [3]:
from openai import OpenAI

PROVIDERS = {
    "openrouter": {
        "api_key": os.getenv("OPENROUTER_API_KEY", "").strip(),
        "base_url": "https://openrouter.ai/api/v1",
        "model": os.getenv("OPENROUTER_MODEL", "meta-llama/llama-3.1-8b-instruct"),
    },
    "groq": {
        "api_key": os.getenv("GROQ_API_KEY", "").strip(),
        "base_url": "https://api.groq.com/openai/v1",
        "model": os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"),
    },
    "deepseek": {
        "api_key": os.getenv("DEEPSEEK_API_KEY", "").strip(),
        "base_url": "https://api.deepseek.com/v1",
        "model": os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),
    },
}

PROVIDER_ORDER = ["groq", "openrouter", "deepseek"]
print("Providers ready:", PROVIDER_ORDER)

Providers ready: ['groq', 'openrouter', 'deepseek']


In [4]:
import gradio as gr

LANGUAGES = {
    "English": "English",
    "Nigerian Pidgin": "Nigerian Pidgin English",
    "Hausa": "Hausa",
    "Yoruba": "Yorùbá",
    "Igbo": "Igbo",
    "French": "French",
    "Spanish": "Spanish",
    "Portuguese": "Portuguese",
    "German": "German",
    "Arabic": "Arabic",
    "Hindi": "Hindi",
    "Swahili": "Swahili",
    "Chinese (Simplified)": "Simplified Chinese",
    "Japanese": "Japanese",
}

HISTORY_FORMAT = "messages"  # Gradio 6.x always uses this
print(f"Gradio {gr.__version__} | {len(LANGUAGES)} languages | format={HISTORY_FORMAT}")

Gradio 6.27.0 | 14 languages | format=messages


In [5]:
def build_prompt(language, level):
    return (
        f"You are BeGinQode, a patient coding tutor for absolute beginners.\n"
        f"Reply in {language}. Learner level: {level}.\n\n"
        f"HOW TO ANSWER:\n"
        f"1. Start with ONE plain-English sentence: what the code does overall.\n"
        f"2. Then explain the code line-by-line, one line per bullet.\n"
        f"3. Then show the fixed or example code in a single ```python block.\n"
        f"4. End with a short '⚠️ Watch out for' note about one common mistake.\n\n"
        f"FORMATTING RULES (VERY IMPORTANT):\n"
        f"- NEVER use markdown tables (no | ... | rows). Tables confuse beginners.\n"
        f"- Use short bullet points and plain sentences instead.\n"
        f"- Keep each bullet to one idea, one line if possible.\n"
        f"- Use **bold** only for key terms (like variable, function, return).\n"
        f"- Keep total answer under 250 words unless the user asks for more.\n"
        f"- No jargon without a one-line definition right after it.\n\n"
        f"CODE RULES:\n"
        f"- Wrap all code in ```language fenced blocks.\n"
        f"- Add inline comments (# like this) explaining each line.\n"
        f"- If fixing a bug: show the broken line, then the fixed line, then why.\n\n"
        f"TONE:\n"
        f"- Warm, encouraging, no condescension.\n"
        f"- If the user's question is unclear, ask ONE clarifying question first.\n"
    )


def call_llm(messages, temperature, max_tokens):
    last_error = None
    for name in PROVIDER_ORDER:
        cfg = PROVIDERS[name]
        if not cfg["api_key"]:
            last_error = f"{name}: no key"
            continue
        try:
            client = OpenAI(api_key=cfg["api_key"], base_url=cfg["base_url"])
            resp = client.chat.completions.create(
                model=cfg["model"],
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
                extra_body={"reasoning_effort": "low"},
            )
            return resp.choices[0].message.content, f"✅ {name}"
        except Exception as e:
            last_error = f"{name}: {str(e)[:120]}"
    return f"⚠️ All providers failed.\n\n{last_error}", "❌ failed"


print("✅ Prompt + caller ready (no tables, bullet-first).")

✅ Prompt + caller ready (no tables, bullet-first).


In [6]:
CSS = """
:root{--bg:#0d0b14;--surf:#16121f;--pur:#8a4fff;--purs:#a175ff;--txt:#e9e4f5;--mut:#b8a8e0}

/* Page */
.gradio-container{background:var(--bg)!important;color:var(--txt)!important;max-width:1050px!important;margin:auto!important;font-family:'Inter','Segoe UI',sans-serif!important}

/* Header */
#bq-h{text-align:center;padding:22px 0 10px;border-bottom:1px solid rgba(138,79,255,.25);margin-bottom:18px}
#bq-h h1{color:var(--purs);letter-spacing:3px;font-weight:700;font-size:2.1rem;margin:0;text-shadow:0 0 20px rgba(138,79,255,.6)}
#bq-h p{color:var(--mut);margin:6px 0 0;font-size:.95rem}

/* Chat window */
#bq-chat{background:var(--surf)!important;border:1px solid rgba(138,79,255,.3)!important;border-radius:14px!important}

/* User bubble */
#bq-chat .message.user{background:linear-gradient(135deg,#5a2ecc,#8a4fff)!important;color:#fff!important;border-radius:14px 14px 4px 14px!important}
#bq-chat .message.user *{color:#fff!important}

/* Bot bubble — background + border */
#bq-chat .message.bot{background:#1e1830!important;border:1px solid rgba(138,79,255,.3)!important;border-radius:14px 14px 14px 4px!important}

/* Bot bubble — TEXT (this is the critical part) */
#bq-chat .message.bot,
#bq-chat .message.bot *,
#bq-chat .message.bot p,
#bq-chat .message.bot li,
#bq-chat .message.bot span,
#bq-chat .message.bot strong,
#bq-chat .message.bot em,
#bq-chat .message.bot b,
#bq-chat .message.bot i,
#bq-chat .message.bot ul,
#bq-chat .message.bot ol,
#bq-chat .message.bot h1,
#bq-chat .message.bot h2,
#bq-chat .message.bot h3,
#bq-chat .message.bot h4,
#bq-chat .message.bot blockquote{
    color:#e9e4f5 !important;
}

/* Inline code in bot bubble */
#bq-chat .message.bot code{
    background:#0a0812!important;
    color:#d6c9ff!important;
    padding:2px 6px!important;
    border-radius:5px!important;
    font-family:'Consolas',monospace!important;
    font-size:.9em!important;
}

/* Code blocks */
#bq-chat pre,
#bq-chat pre code{
    background:#0a0812!important;
    color:#d6c9ff!important;
    border-radius:8px!important;
    font-family:'Consolas',monospace!important;
    padding:12px!important;
    border:1px solid rgba(138,79,255,.35)!important;
    overflow-x:auto!important;
    display:block!important;
}
#bq-chat pre *{color:#d6c9ff!important}

/* Links */
#bq-chat .message.bot a{color:#a175ff!important;text-decoration:underline!important}

/* Bullet markers */
#bq-chat .message.bot li::marker{color:#a175ff!important}

/* Buttons */
button.primary{background:linear-gradient(135deg,#6a35d6,#8a4fff)!important;border:none!important;color:#fff!important;font-weight:600!important}

/* Inputs */
input,textarea,.gr-dropdown,select{background:var(--surf)!important;color:var(--txt)!important;border:1px solid rgba(138,79,255,.35)!important;border-radius:10px!important}

/* Hide footer */
footer{display:none!important}
"""
print("✅ CSS ready (fixed contrast).")

✅ CSS ready (fixed contrast).


In [7]:
TEMPERATURE = 0.3
MAX_TOKENS = 4096

def respond(user_msg, history, lang_label, level_label):
    if not user_msg or not user_msg.strip():
        return history, ""
    if len(user_msg) > 8000:
        err = "⚠️ Message too long. Please split it (max ~8000 chars)."
        return history + [{"role":"user","content":user_msg[:200]+"…"},
                          {"role":"assistant","content":err}], ""
    try:
        lang = LANGUAGES.get(lang_label, "English")
        messages = [{"role":"system","content":build_prompt(lang, level_label)}]
        for t in history:
            if isinstance(t, dict) and "role" in t:
                messages.append({"role":t["role"],"content":t["content"]})
        messages.append({"role":"user","content":user_msg})

        reply, status = call_llm(messages, TEMPERATURE, MAX_TOKENS)
        reply = f"{reply}\n\n_— {status}_"
    except Exception as e:
        reply = f"⚠️ Error: `{type(e).__name__}` — `{str(e)[:200]}`"

    history = history + [{"role":"user","content":user_msg},
                         {"role":"assistant","content":reply}]
    return history, ""


def reset():
    return [], ""

print("✅ Handler ready.")

✅ Handler ready.


In [8]:
with gr.Blocks(title="BeGinQode") as demo:
    gr.HTML('<div id="bq-h"><h1>BeGinQode</h1><p>Understand code better with AI</p></div>')

    with gr.Row():
        lang = gr.Dropdown(choices=list(LANGUAGES.keys()), value="English",
                           label="🌍 Reply Language", scale=2)
        level = gr.Dropdown(
            choices=["Absolute beginner (no coding experience)",
                     "Beginner (knows basics of one language)",
                     "Intermediate (comfortable with functions & loops)"],
            value="Absolute beginner (no coding experience)",
            label="🎓 My level", scale=2)

    chat = gr.Chatbot(elem_id="bq-chat", height=520, label="BeGinQode Chat")

    with gr.Row():
        msg = gr.Textbox(placeholder="Paste code, describe a bug, or ask how something works…",
                         scale=8, lines=2, container=False)
        send = gr.Button("Send", variant="primary", scale=1)
        clear = gr.Button("Reset", scale=1)

    gr.Examples(
        examples=[["Explain this Python code:\n\ndef add(a, b):\n    return a + b"],
                  ["Why do I get: TypeError: 'int' object is not subscriptable"],
                  ["Write a function that reverses a string in JavaScript."],
                  ["What is a variable? Explain like I'm 5."]],
        inputs=msg, label="💡 Try one of these")

    send.click(respond, [msg, chat, lang, level], [chat, msg])
    msg.submit(respond, [msg, chat, lang, level], [chat, msg])
    clear.click(reset, None, [chat, msg])

print("✅ UI built.")

✅ UI built.


In [9]:
try:
    demo.close()
except Exception:
    pass

demo.launch(server_name="127.0.0.1", server_port=7861,
            share=False, css=CSS, show_error=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
